# LevelForge on Google Colab

Runs the full pipeline (BC warm-start → DQN training → GA co-evolution → evaluation & plots) without installing anything on your PC.

**Runtime:** `Runtime → Change runtime type → GPU` (T4) — recommended, though gains are moderate because the environment is a Python loop.

**Get the code (pick ONE):**
- **A. Google Drive (easiest, no GitHub account needed):** upload `LevelForge.zip` to your Drive, then run cell 2a.
- **B. GitHub:** create a repo at github.com, push the folder, then use cell 2b.


In [ ]:
# 2a. Mount Drive and unzip the project (Python zipfile — no `unzip` needed)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!rm -rf /content/levelforge && mkdir -p /content/levelforge
import zipfile, os
with zipfile.ZipFile('/content/drive/MyDrive/LevelForge.zip') as z:
    z.extractall('/content/levelforge')
print(os.listdir('/content/levelforge/LevelForge'))


In [ ]:
# 2b. OR clone from GitHub (replace URL)
# !git clone https://github.com/<YOU>/LevelForge.git /content/levelforge/LevelForge
# (path must be /content/levelforge/LevelForge either way)
!test -d /content/levelforge/LevelForge && echo FOUND || echo NOT_FOUND


In [ ]:
import sys, os
LF = '/content/levelforge/LevelForge'
assert os.path.isdir(LF), 'LevelForge not found — fix the path from cell 2a/2b'
os.chdir(LF)
print('cwd:', os.getcwd())
!ls


In [ ]:
# Make levelforge importable WITHOUT pip (pip's editable install is unreliable on Colab).
# PYTHONPATH is inherited by every `!python ...` subprocess we launch afterwards.
import os
os.chdir('/content/levelforge/LevelForge')
os.environ['PYTHONPATH'] = '/content/levelforge/LevelForge/src'
import sys
sys.path.insert(0, '/content/levelforge/LevelForge/src')
import levelforge
print('levelforge', levelforge.__version__, 'from', levelforge.__file__)
!python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"


In [ ]:
# Sanity check: run the test suite FROM the project root (it gets PYTHONPATH=src from pyproject.toml).
%cd /content/levelforge/LevelForge
!python -m pytest -q tests


In [ ]:
# Quick end-to-end demo
!python -m levelforge demo --size 8 --seed 42


## Train (the part that matters)
CPU Colab: keep sizes small; anything above 8×8 with 2+ boxes is serious compute.

Order: 7×7 1-box (can actually learn) → 8×8 3-box (research-grade).


In [ ]:
# BC warm start (imitation from the generator's own solutions)
!python -m levelforge pretrain-bc --size 7 --n-boxes 1 --levels 400 --epochs 12 --arch cnn --out runs/bc7 --seed 0


In [ ]:
# DQN training with potential-based shaping, warm-started from BC
!python -m levelforge train-baseline --size 7 --n-boxes 1 --episodes 1500 --pool 30 \
  --max-steps 150 --shaping --init-from runs/bc7/bc_policy.pt \
  --out runs/train7 --seed 7


In [ ]:
# GA co-evolution (adversarial curriculum) starting from the BC policy
!python -m levelforge evolve --generations 8 --population 16 --size 7 --n-boxes 1 \
  --generator ga --init-from runs/bc7/bc_policy.pt \
  --out runs --name ga7 --seed 0 --pool 25 --retrain-episodes 60


In [ ]:
# Ablation: same budget, random levels instead of GA breeding (baselines for your report)
!python -m levelforge evolve --generations 8 --population 16 --size 7 --n-boxes 1 \
  --generator random --init-from runs/bc7/bc_policy.pt \
  --out runs --name rand7 --seed 0 --pool 25 --retrain-episodes 60


In [ ]:
# Final evaluation on a held-out set
import glob, json, os
for ckpt in sorted(glob.glob('runs/ga7/checkpoint_gen_*.pt'))[-1:]:
    print(ckpt)
    print(os.path.basename(ckpt))
!python -m levelforge evaluate --checkpoint runs/ga7/checkpoint_gen_7.pt --out runs/ga7
!cat runs/ga7/summary.json || true


## Plots & figures
For the report: success-rate curve and failure breakdown, plus a rendered evolved level.


In [ ]:
import json, matplotlib.pyplot as plt

def load_metrics(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

fig, ax = plt.subplots(figsize=(8, 4))
for name in ['ga7', 'rand7']:
    m = load_metrics(f'runs/{name}/metrics.jsonl')
    ax.plot([r['generation'] for r in m], [r.get('heldout_success_rate', 0) for r in m], marker='o', label=name)
ax.set_xlabel('Generation'); ax.set_ylabel('Held-out success rate')
ax.set_title('GA-evolved vs random-level curriculum'); ax.legend(); ax.grid(alpha=.3)
plt.show()


In [ ]:
# Render the hardest evolved level and show it inline
!python -m levelforge render --level runs/ga7/levels_gen_7/hard_0.json --out runs/ga7/hard_0.png --title 'Evolved adversarial level (gen 7)'
from IPython.display import Image, display
display(Image(filename='runs/ga7/hard_0.png'))


In [ ]:
# Sweep failure types across generations (for the failure-discovery-efficiency metric)
import json
for name in ['ga7', 'rand7']:
    pts = [(r['generation'], r.get('population_avg_failure_rate'), r.get('population_diversity'), r.get('solvability_yield')) for r in load_metrics(f'runs/{name}/metrics.jsonl')]
    print(name)
    for g, f, d, sy in pts:
        print(f'  gen {g}: pop failure={f:.2f}  diversity={d:.3f}  SY={sy if sy is None else round(sy, 2)}')


In [ ]:
# A* solution length (a sanity anchor for your report's 'agent vs optimal' gap)
!python -m levelforge solve --level runs/ga7/levels_gen_7/hard_0.json --max-nodes 500000


In [ ]:
# Download all results
!apt-get -qq install zip > /dev/null 2>&1
!cd /content/levelforge/LevelForge && zip -qr results.zip runs
from google.colab import files
files.download('results.zip')


## Notes / gotchas
- The **GUI viewer won't work** in Colab — use `render` (PNG), `solve` or `watch --text`.
- Free Colab disconnects after ~90 min idle; run short training stages and download `results.zip` early.
- `pip install -e .` is safe here; if it ever complains, fall back to `import os, sys; sys.path.append('src')` and run `python -m levelforge ...` directly.
- If you exceed GPU RAM, add `--arch mlp` to `pretrain-bc` or drop `--size`.
- Results are noisy — run 3 seeds for the headline comparison and report mean ± std.
